# 🎸 Given — Fuyu no Hanashi (冬のはなし) — AI Русская версия
### Оригинальный вокал Мафую Сато (Shogo Yano / Centimillimental) поет русский перевод песни!

**Инструкция: Нажмите Среда выполнения -> Выполнить все (Runtime -> Run all).**

Пайплайн выполнится полностью автоматически за **~8-10 минут** на ускорителе Tesla T4 GPU:
1. **Окружение**: Развертывание Applio RVC v2, нейросети Demucs и весов (RMVPE, ContentVec, HiFi-GAN 40k).
2. **Разделение (Demucs)**: Извлечение кристально чистого акапелла-вокала японского автора и русской партии.
3. **Обучение (RVC v2)**: Извлечение частот связок (RMVPE) и обучение модели голоса Мафую (100 эпох).
4. **AI-Пение**: Нейросеть берет русский вокал и перепевает его 100% аутентичным тембром Мафую.
5. **Сведение и мастеринг**: Сведение AI-вокала со студийным инструменталом в готовый MP3 (320 kbps).
6. **Результат**: Трек играет прямо в браузере и автоматически скачивается на устройство!

## Шаг 1. Подготовка GPU среды, Applio и моделей

In [ ]:
!nvidia-smi

import os, subprocess
%cd /content
if not os.path.exists("/content/Applio"):
    !git clone --depth 1 https://github.com/IAHispano/Applio.git /content/Applio
%cd /content/Applio

# Установка aria2, ffmpeg и библиотек
!apt-get update -y -qq && apt-get install -y -qq aria2 ffmpeg portaudio19-dev
!pip install -q "numpy<2.4"
!pip install -q faiss-cpu torchcrepe torchfcpe stftpitchshift soxr einops demucs onnxruntime
!pip install -q "numpy<2.4"

# Папки для моделей
os.makedirs("rvc/models/predictors", exist_ok=True)
os.makedirs("rvc/models/embedders/contentvec", exist_ok=True)
os.makedirs("rvc/models/pretraineds/hifi-gan", exist_ok=True)
os.makedirs("assets/datasets/Mafuyu", exist_ok=True)

print("Быстрое скачивание весов RMVPE, ContentVec и HiFi-GAN...")
models = [
    ("https://huggingface.co/IAHispano/Applio/resolve/main/Resources/predictors/rmvpe.pt", "rvc/models/predictors/rmvpe.pt"),
    ("https://huggingface.co/IAHispano/Applio/resolve/main/Resources/embedders/contentvec/pytorch_model.bin", "rvc/models/embedders/contentvec/pytorch_model.bin"),
    ("https://huggingface.co/IAHispano/Applio/resolve/main/Resources/embedders/contentvec/config.json", "rvc/models/embedders/contentvec/config.json"),
    ("https://huggingface.co/IAHispano/Applio/resolve/main/Resources/pretrained_v2/f0G40k.pth", "rvc/models/pretraineds/hifi-gan/f0G40k.pth"),
    ("https://huggingface.co/IAHispano/Applio/resolve/main/Resources/pretrained_v2/f0D40k.pth", "rvc/models/pretraineds/hifi-gan/f0D40k.pth"),
]
for url, out_path in models:
    if not os.path.exists(out_path):
        out_dir = os.path.dirname(out_path)
        out_name = os.path.basename(out_path)
        !aria2c -x 16 -s 16 -k 1M -q -d "{out_dir}" -o "{out_name}" "{url}"

!cp assets/config_template.json assets/config.json
import torch
print(f"✓ Окружение готово! GPU: {torch.cuda.get_device_name(0)}")


## Шаг 2. Загрузка треков Given и русского вокала

In [ ]:
import os, shutil
%cd /content
if not os.path.exists("/content/piper-voice"):
    !git clone https://github.com/Holki9/piper-voice.git /content/piper-voice
else:
    !cd /content/piper-voice && git pull origin main

# Проверяем файлы
songs_dir = "/content/piper-voice/songs"
jp_song = f"{songs_dir}/Given_Fuyunohanashi.mp3"
jp_marutsuke = f"{songs_dir}/Given_Marutsuke.mp3"
ru_song = f"{songs_dir}/fuyu_russian_trimmed.mp3"

assert os.path.exists(jp_song), f"Не найден {jp_song}"
assert os.path.exists(ru_song), f"Не найден {ru_song}"
print("✓ Все необходимые аудиофайлы на месте!")
!ls -lh /content/piper-voice/songs/

## Шаг 3. Нейросетевое разделение дорожек (Demucs v4)

In [ ]:
import glob, os, shutil
%cd /content

print("1. Выделяем вокал японского автора (Given - Fuyunohanashi)...")
!demucs --two-stems=vocals /content/piper-voice/songs/Given_Fuyunohanashi.mp3 -o /content/sep_jp

if os.path.exists("/content/piper-voice/songs/Given_Marutsuke.mp3"):
    print("2. Выделяем дополнительный вокал автора (Given - Marutsuke)...")
    !demucs --two-stems=vocals /content/piper-voice/songs/Given_Marutsuke.mp3 -o /content/sep_marutsuke

print("3. Выделяем чистый вокал русской дорожки...")
!demucs --two-stems=vocals /content/piper-voice/songs/fuyu_russian_trimmed.mp3 -o /content/sep_ru

# Копируем японский вокал в датасет для обучения голоса Мафую
os.makedirs("/content/Applio/assets/datasets/Mafuyu", exist_ok=True)
jp_vocals = glob.glob("/content/sep_jp/**/vocals.wav", recursive=True)[0]
shutil.copy2(jp_vocals, "/content/Applio/assets/datasets/Mafuyu/mafuyu_fuyu.wav")

maru_vocals = glob.glob("/content/sep_marutsuke/**/vocals.wav", recursive=True)
if maru_vocals:
    shutil.copy2(maru_vocals[0], "/content/Applio/assets/datasets/Mafuyu/mafuyu_maru.wav")

ru_vocals = glob.glob("/content/sep_ru/**/vocals.wav", recursive=True)[0]
ru_instr = glob.glob("/content/sep_ru/**/no_vocals.wav", recursive=True)[0]
shutil.copy2(ru_vocals, "/content/russian_lead_vocals.wav")
shutil.copy2(ru_instr, "/content/instrumental_track.wav")

print("✓ Разделение завершено! Датасет голоса автора подготовлен:")
!ls -lh /content/Applio/assets/datasets/Mafuyu/


## Шаг 4. Обучение модели RVC v2 на голосе Мафую Сато (100 эпох)

In [ ]:
%cd /content/Applio

# 1. Нарезка и ресемплинг датасета автора
print("Нарезка аудио...")
!python core.py preprocess --model-name Mafuyu --dataset-path assets/datasets/Mafuyu --sample-rate 40000 --cpu-cores 2

# 2. Извлечение питча (RMVPE) и эмбеддингов (ContentVec)
print("Извлечение питча связок (RMVPE)...
")
!python core.py extract --model-name Mafuyu --sample-rate 40000 --f0-method rmvpe --embedder-model contentvec

# 3. Быстрое обучение модели (100 эпох, ~5-6 минут)
print("Запуск обучения модели голоса Мафую Сато на GPU...")
!python core.py train --model-name Mafuyu --save-every-epoch 25 --total-epoch 100 --sample-rate 40000 --batch-size 8 --gpu 0 --vocoder HiFi-GAN --save-only-latest --pretrained

# Проверка
import glob
pths = glob.glob("/content/Applio/logs/Mafuyu/*.pth")
assert len(pths) > 0, "Ошибка: модель Mafuyu.pth не создана!"
print("✓ Модель голоса автора успешно обучена:")
!ls -lh logs/Mafuyu/


## Шаг 5. Нейросетевая конвертация вокала в голос автора (RVC)

In [ ]:
import glob, os, shlex, subprocess
%cd /content/Applio

!ls -t /content/Applio/logs/Mafuyu/*.pth 2> /dev/null | head -n 1 > /tmp/pth.txt
pth_file = open("/tmp/pth.txt").read().strip()
!ls -t /content/Applio/logs/Mafuyu/*.index 2> /dev/null | head -n 1 > /tmp/index.txt
index_file = open("/tmp/index.txt").read().strip()

print(f"Используем модель: {pth_file}")
print(f"Используем индекс: {index_file}")

print("Перепеваем русский вокал аутентичным тембром Мафую...")
infer_cmd = [
    "python", "core.py", "infer",
    "--pitch", "0",
    "--volume-envelope", "0.85",
    "--index-rate", "0.75",
    "--protect", "0.33",
    "--f0-method", "rmvpe",
    "--input-path", "/content/russian_lead_vocals.wav",
    "--output-path", "/content/mafuyu_russian_vocals.wav",
    "--pth-path", pth_file,
    "--export-format", "WAV",
    "--embedder-model", "contentvec"
]
if index_file and os.path.exists(index_file):
    infer_cmd += ["--index-path", index_file]

!{shlex.join(infer_cmd)}
assert os.path.exists("/content/mafuyu_russian_vocals.wav"), "Ошибка конвертации!"
print("✓ Голос успешно перепет!")


## Шаг 6. Сведение со студийным инструменталом и сохранение

In [ ]:
import subprocess, os
from IPython.display import Audio, display
from google.colab import files

out_file = "/content/Given_Fuyunohanashi_Russian_AI.mp3"

# Мастеринг: балансировка уровней вокала и музыки
mix_cmd = [
    "ffmpeg", "-y",
    "-i", "/content/mafuyu_russian_vocals.wav",
    "-i", "/content/instrumental_track.wav",
    "-filter_complex", "[0:a]volume=1.25,acompressor=threshold=-12dB:ratio=3:attack=5:release=50[v];[1:a]volume=0.95[m];[v][m]amix=inputs=2:duration=first:dropout_transition=2",
    "-b:a", "320k",
    out_file
]
subprocess.run(mix_cmd, check=True)

print("
🎉 ГОТОВО! Песня Given — Fuyu no Hanashi на русском голосом Мафую!")
display(Audio(out_file))

print("Скачивание готового трека...")
files.download(out_file)
